# Notebook 02 — The Divergence (∇·F)

Divergence turns a **vector field** into a **scalar field**. At every point,
∇·F measures how much the field "spreads outward" from that point.

$$\nabla \cdot \mathbf{F} = \frac{\partial F_x}{\partial x} + \frac{\partial F_y}{\partial y}$$

Intuition: imagine the field is the velocity of a fluid.

- **∇·F > 0** at a point → fluid is being created there (a *source*, like a tap).
- **∇·F < 0** → fluid is disappearing there (a *sink*, like a drain).
- **∇·F = 0** everywhere → the fluid is *incompressible*; whatever flows in
  flows out, and the field is called **solenoidal**.


In [ ]:
# Make src/ importable from inside notebooks/
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
%matplotlib inline

from src import operators as ops
from src import visualizers as viz


## 1. By hand — a radial source field

Let $\mathbf{F}(x, y) = (x, y)$. This is the **outward radial field**: at every
point, $\mathbf{F}$ points away from the origin with magnitude equal to the
distance from the origin. We expect this to look like a source everywhere.

- $\dfrac{\partial F_x}{\partial x} = \dfrac{\partial x}{\partial x} = 1$
- $\dfrac{\partial F_y}{\partial y} = \dfrac{\partial y}{\partial y} = 1$
- $\nabla \cdot \mathbf{F} = 1 + 1 = 2$ — constant, positive everywhere. ✓

## 2. By hand — a non-trivial example

Let $\mathbf{F}(x, y) = (xy,\ y^2 - x)$.

- $\dfrac{\partial F_x}{\partial x} = y$
- $\dfrac{\partial F_y}{\partial y} = 2y$
- $\nabla \cdot \mathbf{F} = 3y$

Net source above the x-axis, net sink below.


## 3. SymPy verification

In [ ]:
x, y = sp.symbols('x y')
Fx_sym = x * y
Fy_sym = y**2 - x

div_sym = sp.diff(Fx_sym, x) + sp.diff(Fy_sym, y)
print("∇·F =", div_sym)


## 4. NumPy on a grid

In [ ]:
x_vals = np.linspace(-2, 2, 200)
y_vals = np.linspace(-2, 2, 200)
X, Y = np.meshgrid(x_vals, y_vals)
dx, dy = x_vals[1] - x_vals[0], y_vals[1] - y_vals[0]

Fx = X * Y
Fy = Y**2 - X

div_num = ops.divergence_2d(Fx, Fy, dx, dy)
div_exact = 3 * Y

print(f"Max error: {np.max(np.abs(div_num - div_exact)):.2e}")


## 5. Visualization

In [ ]:
fig = viz.plot_vector_with_divergence(
    X, Y, Fx, Fy, div_num,
    title=r"$\mathbf{F} = (xy,\ y^2 - x)$  —  $\nabla\cdot\mathbf{F} = 3y$",
)
plt.show()


**Reading this plot.** Black streamlines trace the flow. The
background is the divergence: red where flow is being created, blue where
it's being absorbed, white near the x-axis where divergence is zero. The
streamlines visibly emerge from the red regions and converge into the blue
ones.

## 6. A solenoidal example: divergence-free flow

In [ ]:
# Pure rotation field: F = (-y, x)
Fx_rot = -Y
Fy_rot = X

div_rot = ops.divergence_2d(Fx_rot, Fy_rot, dx, dy)
print(f"Max |divergence| of rotation field: {np.max(np.abs(div_rot)):.2e}")

fig = viz.plot_vector_with_divergence(
    X, Y, Fx_rot, Fy_rot, div_rot,
    title=r"Rotation field $\mathbf{F} = (-y, x)$  —  divergence-free",
)
plt.show()


The streamlines are circles, and the divergence is essentially zero
everywhere (small numerical noise from the finite-difference scheme at the
boundary). This is a **solenoidal** field — flow that only swirls, never
sources or sinks.

## 7. Exercise

For each field, predict the sign of the divergence at $(1, 1)$ before computing it:

1. $\mathbf{F} = (x^2,\ y^2)$
2. $\mathbf{F} = (-x,\ -y)$  *(inward radial)*
3. $\mathbf{F} = (y,\ -x)$
